In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

df = pd.read_csv("train.csv")
df.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [3]:
# melakukan prediksi sederhana
# kelompokkan terlebih dahulu

X = df.drop(columns=["PitNextLap"])
y = df.PitNextLap

X.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0


In [4]:
# pisahkan menggunakan split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

X_train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
215889,215889,D017,HARD,Pre-Season Testing,2023,0,28,2,10.0,12,92.878,-0.097,-20.820,0.538462,0.0
204847,204847,PEA,INTERMEDIATE,Canadian Grand Prix,2024,0,25,1,25.0,10,91.448,-28.820,-31.977,0.328947,3.0
118844,118844,D242,SOFT,Dutch Grand Prix,2022,0,6,1,7.0,2,77.664,-36.765,-45.569,0.083333,-2.0
258088,258088,D020,HARD,Las Vegas Grand Prix,2025,0,28,2,18.0,8,95.822,-1.800,-201.840,0.368421,4.0
90334,90334,HAD,HARD,Emilia Romagna Grand Prix,2024,0,38,2,21.0,7,82.534,-11.398,-136.539,0.493506,-3.0


In [5]:
# deteksi apakah ada kolom yang kosong pada train dan test
null_cols = [col for col in df.columns if df[col].isnull().any()]
print(null_cols)
# tidak terdapat column yang kosong
# bisa lewatkan untuk step 
print("tidak memiliki data yang kosong")

[]
tidak memiliki data yang kosong


In [6]:
# melakukan pembacaan kategori data
object_cols = X_train.select_dtypes(['object'])
# pisahkan antara high cardinality data dan low cardinality data
# low_cardinality_cols = [col for col in ]
print(object_cols.columns)
# mencari mana yang mempunyai low dan high cardinality
object_nunique = list(map(lambda col: X_train[col].nunique(), object_cols))
d = dict(zip(object_cols, object_nunique))

sorted(d.items(), key=lambda x:x[1])

Index(['Driver', 'Compound', 'Race'], dtype='str')


C:\Users\740195\AppData\Local\Temp\ipykernel_28720\166770103.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = X_train.select_dtypes(['object'])


[('Compound', 5), ('Race', 26), ('Driver', 875)]

In [7]:
# jadikan compound sebagai ordinal encoding
# mendefinisikan urutan encoder
compound_category = [['INTERMEDIATE', 'HARD', 'MEDIUM', 'SOFT', 'WET']]
my_ordinal_encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value', 
    unknown_value=-1,
    categories=compound_category
    )

X_train.head()
# melakukan drop terhadap data yang jelek
bad_label_cols = ['Driver']
good_label_cols_for_H = ['Compound']

label_X_train = X_train.drop(bad_label_cols, axis=1)
label_X_test = X_test.drop(bad_label_cols, axis=1)

print(display(label_X_train.head()))
# bagian driver sudah di drop

,id,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
215889,215889,HARD,Pre-Season Testing,2023,0,28,2,10.0,12,92.878,-0.097,-20.820,0.538462,0.0
204847,204847,INTERMEDIATE,Canadian Grand Prix,2024,0,25,1,25.0,10,91.448,-28.820,-31.977,0.328947,3.0
118844,118844,SOFT,Dutch Grand Prix,2022,0,6,1,7.0,2,77.664,-36.765,-45.569,0.083333,-2.0
258088,258088,HARD,Las Vegas Grand Prix,2025,0,28,2,18.0,8,95.822,-1.800,-201.840,0.368421,4.0
90334,90334,HARD,Emilia Romagna Grand Prix,2024,0,38,2,21.0,7,82.534,-11.398,-136.539,0.493506,-3.0


None


In [8]:
print(X_train['Compound'].unique())

<StringArray>
['HARD', 'INTERMEDIATE', 'SOFT', 'MEDIUM', 'WET']
Length: 5, dtype: str


In [9]:
# melakukan encoding ordinal pada good code
label_X_train[good_label_cols_for_H] = my_ordinal_encoder.fit_transform(X_train[good_label_cols_for_H])
label_X_test[good_label_cols_for_H] = my_ordinal_encoder.transform(X_test[good_label_cols_for_H])

print(display(label_X_train.head()))
# berhasil melakukan ordinal encoder yang sudah di siapkan, tinggal bagian race untuk di hot encoding

,id,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
215889,215889,1.0,Pre-Season Testing,2023,0,28,2,10.0,12,92.878,-0.097,-20.820,0.538462,0.0
204847,204847,0.0,Canadian Grand Prix,2024,0,25,1,25.0,10,91.448,-28.820,-31.977,0.328947,3.0
118844,118844,3.0,Dutch Grand Prix,2022,0,6,1,7.0,2,77.664,-36.765,-45.569,0.083333,-2.0
258088,258088,1.0,Las Vegas Grand Prix,2025,0,28,2,18.0,8,95.822,-1.800,-201.840,0.368421,4.0
90334,90334,1.0,Emilia Romagna Grand Prix,2024,0,38,2,21.0,7,82.534,-11.398,-136.539,0.493506,-3.0


None


In [10]:
# melakukan one hot encoding pada column race
from sklearn.preprocessing import OneHotEncoder

low_cardinality_cols = ['Race']

OH_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(label_X_train[low_cardinality_cols]))
OH_cols_test = pd.DataFrame(OH_encoder.transform(label_X_test[low_cardinality_cols]))

# masukkan kembali index nya
OH_cols_train.index = label_X_train.index
OH_cols_test.index = label_X_test.index

print(display(OH_cols_train.head()))

# remove column yang digunakan
num_X_train = label_X_train.drop(low_cardinality_cols, axis=1)
num_X_test = label_X_test.drop(low_cardinality_cols, axis=1)
# maka sudah tidak ada kolomnya
# gabungkan keduanya kembali
display(num_X_train)

OH_X_train = pd.concat([OH_cols_train, num_X_train], axis=1)
OH_X_test = pd.concat([OH_cols_test, num_X_test], axis=1)

# pastikan columnya berbentuk string
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_test.columns = OH_X_test.columns.astype(str)

display(OH_X_train.head())

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
215889,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
204847,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
118844,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
258088,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
90334,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


None


,id,Compound,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
215889,215889,1.0,2023,0,28,2,10.0,12,92.878,-0.097,-20.820,0.538462,0.0
204847,204847,0.0,2024,0,25,1,25.0,10,91.448,-28.820,-31.977,0.328947,3.0
118844,118844,3.0,2022,0,6,1,7.0,2,77.664,-36.765,-45.569,0.083333,-2.0
258088,258088,1.0,2025,0,28,2,18.0,8,95.822,-1.800,-201.840,0.368421,4.0
90334,90334,1.0,2024,0,38,2,21.0,7,82.534,-11.398,-136.539,0.493506,-3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
73349,73349,2.0,2022,1,27,2,9.0,12,99.928,-3.876,-1.561,0.375000,0.0
371403,371403,2.0,2023,0,1,1,1.0,13,117.922,0.000,0.000,0.020000,0.0
312201,312201,2.0,2024,0,42,2,42.0,12,80.849,-1.716,3.001,0.552632,5.0
267336,267336,2.0,2022,0,6,1,6.0,14,97.577,-7.081,-6.082,0.105263,2.0


,0,1,2,3,4,5,6,7,8,9,...,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
215889,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,28,2,10.0,12,92.878,-0.097,-20.820,0.538462,0.0
204847,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0,25,1,25.0,10,91.448,-28.820,-31.977,0.328947,3.0
118844,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0,6,1,7.0,2,77.664,-36.765,-45.569,0.083333,-2.0
258088,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,28,2,18.0,8,95.822,-1.800,-201.840,0.368421,4.0
90334,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,38,2,21.0,7,82.534,-11.398,-136.539,0.493506,-3.0


In [ ]:
X_train = OH_X_train
X_test = OH_X_test

# my_model = RandomForestClassifier()

# my_model.fit(X_train, y_train)
# 2. Definisikan 8 kombinasi parameter yang berbeda
param_combinations = [
    {"n_estimators": 50,  "max_depth": 8,    "class_weight": None,       "min_samples_split": 2, "name": "Model 1 (Light & Standard)"},
    {"n_estimators": 100, "max_depth": 10,   "class_weight": None,       "min_samples_split": 5, "name": "Model 2 (Medium & Regularized)"},
    {"n_estimators": 150, "max_depth": 15,   "class_weight": None,       "min_samples_split": 2, "name": "Model 3 (Deep & Dense)"},
    {"n_estimators": 100, "max_depth": 12,   "class_weight": "balanced", "min_samples_split": 4, "name": "Model 4 (Medium & Balanced)"},
    {"n_estimators": 200, "max_depth": 15,   "class_weight": "balanced", "min_samples_split": 5, "name": "Model 5 (Large & Balanced)"},
    {"n_estimators": 80,  "max_depth": 8,    "class_weight": "balanced", "min_samples_split": 2, "name": "Model 6 (Shallow & Balanced)"},
    {"n_estimators": 200, "max_depth": None, "class_weight": None,       "min_samples_split": 10,"name": "Model 7 (Full Depth & High Split)"},
    {"n_estimators": 150, "max_depth": 20,   "class_weight": "balanced", "min_samples_split": 10,"name": "Model 8 (Deep & Balanced Split)"}
]

results = []

# 3. Looping untuk training dan evaluasi ke-8 model
print("Starting training loop...\n" + "-"*50)
for params in param_combinations:
    # Ambil nama model lalu hapus key 'name' agar tidak error saat dimasukkan ke RandomForest
    model_name = params["name"]
    rf_params = {k: v for k, v in params.items() if k != "name"}
    
    # Inisialisasi model dengan parameter saat ini
    model = RandomForestClassifier(**rf_params, random_state=42, n_jobs=-1)
    
    # Fitting model
    model.fit(X_train, y_train)
    
    # Prediksi pada data validation
    y_pred = model.predict(X_test)
    
    # Hitung F1-score (menggunakan rata-rata macro atau binary sesuai targetmu)
    # Jika targetnya 0 dan 1 biasa, pakai average='binary'
    score = accuracy_score(y_test, y_pred)
    
    results.append({
        "Model": model_name,
        "F1-Score": score,
        "Parameters": rf_params
    })
    print(f"✅ {model_name} finished! Validation F1-Score: {score:.4f}")

# 4. Tampilkan hasil urut dari F1-score tertinggi
df_results = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
print("\n" + "="*50 + "\nRANKING MODEL TERBAIK:\n" + "="*50)
print(df_results[["Model", "F1-Score"]])

# berhasil melakukan fitting

Starting training loop...
--------------------------------------------------
✅ Model 1 (Light & Standard) finished! Validation F1-Score: 0.8673
✅ Model 2 (Medium & Regularized) finished! Validation F1-Score: 0.8771
✅ Model 3 (Deep & Dense) finished! Validation F1-Score: 0.8894
✅ Model 4 (Medium & Balanced) finished! Validation F1-Score: 0.8546
✅ Model 5 (Large & Balanced) finished! Validation F1-Score: 0.8617
✅ Model 6 (Shallow & Balanced) finished! Validation F1-Score: 0.8423
✅ Model 7 (Full Depth & High Split) finished! Validation F1-Score: 0.8949
✅ Model 8 (Deep & Balanced Split) finished! Validation F1-Score: 0.8704

RANKING MODEL TERBAIK:
                               Model  F1-Score
6  Model 7 (Full Depth & High Split)  0.894851
2             Model 3 (Deep & Dense)  0.889397
1     Model 2 (Medium & Regularized)  0.877078
7    Model 8 (Deep & Balanced Split)  0.870372
0         Model 1 (Light & Standard)  0.867252
4         Model 5 (Large & Balanced)  0.861673
3        Model 4 (M

In [14]:
my_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight=None,
    min_samples_split=10
)

my_model.fit(X_train, y_train)

# mengambil score precission
predictions = my_model.predict(X_test)
# mendapatkan score accuracy
accuracy = accuracy_score(y_test, predictions)
report = classification_report(y_test, predictions)

print(f"Hasil accuracy scorenya adalah : {accuracy}")
print(report)

Hasil accuracy scorenya adalah : 0.8953408935646946
              precision    recall  f1-score   support

         0.0       0.93      0.94      0.94     70317
         1.0       0.76      0.70      0.73     17511

    accuracy                           0.90     87828
   macro avg       0.84      0.82      0.83     87828
weighted avg       0.89      0.90      0.89     87828



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Cek kolom mana yang paling dominan
importances = pd.Series(my_model.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False).head(5))

LapTime_Delta             0.111201
TyreLife                  0.104344
Stint                     0.087170
Year                      0.083128
Cumulative_Degradation    0.082566
dtype: float64


In [15]:
# mulai bermain dengan data test
dft = pd.read_csv('test.csv')
display(dft.head())

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,439140,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
1,439141,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
2,439142,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
3,439143,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
4,439144,AND,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0


In [16]:
dft = pd.read_csv('test.csv')

label_test = dft.drop(bad_label_cols, axis=1)
# print(label_test)
label_test[good_label_cols_for_H] = my_ordinal_encoder.transform(dft[good_label_cols_for_H])
label_test.head()
# ordinal encoder dulu
# one hot encoder menggunakan persis seperti di atas
OH_test_cols = pd.DataFrame(OH_encoder.transform(label_test[low_cardinality_cols]))

# lakukan penyamaan index
OH_test_cols.index = label_test.index
display(OH_test_cols.head())
# drop data pada data awal
num_test = label_test.drop(low_cardinality_cols, axis=1)
# gabungkan keduanya
OH_final_X = pd.concat([OH_test_cols, num_test], axis=1)

# pastikan semuanya str
OH_final_X.columns = OH_final_X.columns.astype(str)

display(OH_final_X.head())

# lakukan prediksi
final_test_predict = my_model.predict(OH_final_X)

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


,0,1,2,3,4,5,6,7,8,9,...,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0


In [17]:
# masukkan ke dalam submission
submission = pd.DataFrame({
    'id': dft['id'],
    'PitNextLap': final_test_predict
})

# 4. Simpan ke dalam file CSV tanpa menyertakan indeks baris
submission.to_csv('submission.csv', index=False)

print("File 'submission.csv' berhasil dibuat!")

File 'submission.csv' berhasil dibuat!
